In [ ]:
import pandas as pd
df=pd.read_csv('/content/crime_data1.csv')
df.head()

,Description,Label,crime_types,transformed_tweet
0,The U.N. organization assisting in investigati...,crime,0,The U.N. organization assisting in investigati...
1,RG Kar Rape Case: The body of the trainee doct...,assault,3,RG Kar Rape Case: The body of the trainee doct...
2,The IT hub of India faces a strange kind of pr...,crime,0,The IT hub of India faces a strange kind of pr...
3,Police in California released a video of a tri...,crime,0,Police in California released a video of a tri...
4,Crime Officer fatally shot in a North Carolina...,crime,0,Crime Officer fatally shot in a North Carolina...


In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Encode the target labels (crime_types)
#label_mapping = {label: idx for idx, label in enumerate(df['crime_types'].unique())}
#df['label_encoded'] = df['crime_types'].map(label_mapping)

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['transformed_tweet'].tolist(),
    df['crime_types'].tolist(),
    test_size=0.2,
    random_state=42
)


In [ ]:
# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')


In [ ]:
# Custom Dataset Class
class CrimeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Datasets & DataLoaders
train_dataset = CrimeDataset(train_texts, train_labels, tokenizer)
val_dataset = CrimeDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)


In [ ]:
# Classification Model
class CrimeClassifier(nn.Module):
    def __init__(self, num_classes):
        super(CrimeClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token output
        cls_output = self.dropout(cls_output)
        logits = self.fc(cls_output)
        return logits


In [ ]:
# Model Setup
num_classes = df['crime_types'].nunique()
model = CrimeClassifier(num_classes)



In [ ]:

# Optimizer and Loss
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Training Loop
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['label']

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}, Loss: {avg_loss:.4f}')


Epoch 1, Loss: 1.7561
Epoch 2, Loss: 1.2897
Epoch 3, Loss: 1.0660


In [ ]:
# Evaluation
model.eval()
predictions, true_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        labels = batch['label']

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())
        true_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(true_labels, predictions)
print(f'Validation Accuracy: {accuracy:.4f}')

Validation Accuracy: 0.7500
